In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, random_split, DataLoader
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image

In [2]:
DATA_DIR = Path("../../data")

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [4]:
class ImageDataset(Dataset):
    def __init__(self, files, transform=None):
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        image = Image.open(file).convert("RGB")
        label = 0 if file.stem.startswith("cat.") else 1

        if self.transform:
            image = self.transform(image)

        return image, label

In [5]:
training_files = list(Path(f"{DATA_DIR}/train").glob("*"))

In [6]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [7]:
dataset = ImageDataset(training_files, transform=transform)

In [8]:
train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 20000
Validation: 5000


In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [10]:
import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [11]:
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
epochs = 10

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_loss = float("inf")

In [ ]:
from tqdm.auto import tqdm

for epoch in range(epochs):

    # Train
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    train_progress = tqdm(
        train_loader,
        desc=f"Epoch [{epoch+1}/{epochs}] Train"
    )

    for images, labels in train_progress:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

        train_progress.set_postfix(
            loss=f"{train_loss / train_total:.4f}",
            acc=f"{train_correct / train_total:.4f}"
        )

    train_loss /= train_total
    train_acc = train_correct / train_total

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f}"
    )

    # Validation
    model.eval()

KeyboardInterrupt: 

In [ ]:
val_loss = 0
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        val_correct += (predictions == labels).sum().item()
        val_total += labels.size(0)

val_loss /= val_total
val_acc = val_correct / val_total

# History
history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)

print(
    f"Epoch [{epoch+1}/{epochs}] "
    f"Train Loss: {train_loss:.4f} "
    f"Train Acc: {train_acc:.4f} | "
    f"Val Loss: {val_loss:.4f} "
    f"Val Acc: {val_acc:.4f}"
)

Epoch [30/30] Train Loss: 0.0166 Train Acc: 0.9943 | Val Loss: 1.4560 Val Acc: 0.8132


In [ ]:
test_files = sorted(Path(f"{DATA_DIR}/test").glob("*.jpg"))

In [ ]:
class TestDataset(Dataset):
    def __init__(self, files, transform=None):
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        image = Image.open(file).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, file.name

In [ ]:
test_dataset = TestDataset(
    test_files,
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
model.eval()

predictions = []
filenames = []

with torch.no_grad():
    for images, names in test_loader:
        images = images.to(device)

        outputs = model(images)

        # 고양이(Class 1)일 확률만 추출
        probs = torch.softmax(outputs, dim=1)[:, 1]

        predictions.extend(probs.cpu().numpy())
        filenames.extend(names)

In [ ]:
submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
submission["label"] = predictions

submission.to_csv(f"{DATA_DIR}/submission.csv", index=False)